# Step 2 — Translate with GPT & Gemini

For each non-English sentence (Spanish, French, Italian) call two models via OpenRouter:
- **GPT-5.2-chat** (`openai/gpt-5.2-chat`)
- **Gemini-2.5-flash** (`google/gemini-2.5-flash`)

Both use the **same prompt** and the same API key (OpenRouter).

Outputs saved to:
- `outputs/translations_gpt.csv`
- `outputs/translations_gemini.csv`

Each output has columns: `id | source_lang | source_text | translated_text | model`

In [3]:
import os, time
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(r"c:\Users\Nguyen Ngo\Downloads\Thesis Quynh\.env")

OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY")

# OpenRouter client (works for both GPT and Gemini)
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_KEY,
)

DATA_PATH    = r"c:\Users\Nguyen Ngo\Downloads\Thesis Quynh\data\sentences.csv"
OUT_GPT      = r"c:\Users\Nguyen Ngo\Downloads\Thesis Quynh\outputs\translations_gpt.csv"
OUT_GEMINI   = r"c:\Users\Nguyen Ngo\Downloads\Thesis Quynh\outputs\translations_gemini.csv"

MODELS = {
    "gpt":    "openai/gpt-5.2",
    "gemini": "google/gemini-2.5-flash",
}

SOURCE_LANGS = ["Spanish", "French", "Italian"]

sentences = pd.read_csv(DATA_PATH)
print(f"Loaded {len(sentences)} sentences")
sentences.head(3)


Loaded 200 sentences


,id,English,Spanish,French,Italian
0,1,Cable car gives quick access to spectacular vi...,El telefrico ofrece acceso rpido a vistas es...,Le tlphrique offre un accs rapide  des vu...,La funivia offre un rapido accesso a viste spe...
1,2,A museum at the top in an old fort and also go...,Un museo en la cima de un antiguo fuerte y tam...,Un muse au sommet d'un vieux fort et un bon r...,Un museo in cima in un vecchio forte e anche u...
2,3,Dubrovnik Old Town is magical and nowhere more...,"El casco antiguo de Dubrovnik es mgico, y en ...","La vieille ville de Dubrovnik est magique, et ...",La citt vecchia di Dubrovnik  magica e da ne...


In [4]:
SYSTEM_PROMPT = (
    "You are a professional translator. "
    "Translate the given sentence into English. "
    "Return ONLY the translated sentence — no explanations, no quotes, no extra text."
)

def translate_sentence(text: str, model_id: str, retries: int = 3) -> str:
    """Call OpenRouter and return the English translation."""
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": text},
                ],
                temperature=0,   # deterministic
                max_tokens=256,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)  # exponential back-off
            else:
                return f"ERROR: {e}"

print("translate_sentence() ready.")

translate_sentence() ready.


In [5]:
def run_translation(model_key: str, output_path: str):
    """Translate all source sentences with the given model, skip already done rows."""
    model_id = MODELS[model_key]

    # Resume from existing file if present
    if os.path.exists(output_path):
        done = pd.read_csv(output_path)
        done_ids = set(zip(done["id"], done["source_lang"]))
        print(f"[{model_key}] Resuming — {len(done)} rows already done")
    else:
        done = pd.DataFrame()
        done_ids = set()

    rows = []
    total = len(sentences) * len(SOURCE_LANGS)
    count = 0

    for _, row in sentences.iterrows():
        for lang in SOURCE_LANGS:
            count += 1
            src_text = row[lang]

            # Skip missing source text
            if pd.isna(src_text):
                continue

            # Skip if already translated
            if (row["id"], lang) in done_ids:
                continue

            translation = translate_sentence(src_text, model_id)
            rows.append({
                "id":               row["id"],
                "source_lang":      lang,
                "source_text":      src_text,
                "translated_text":  translation,
                "model":            model_key,
            })

            if count % 20 == 0:
                print(f"  [{model_key}] {count}/{total} ...")

            time.sleep(0.3)  # polite rate limiting

    # Append new rows to existing and save
    new_df = pd.DataFrame(rows)
    final  = pd.concat([done, new_df], ignore_index=True) if len(done) else new_df
    final.to_csv(output_path, index=False, encoding="utf-8")
    print(f"[{model_key}] Done — {len(final)} total rows saved → {output_path}")
    return final

print("run_translation() ready.")

run_translation() ready.


In [6]:
# ── Run GPT ────────────────────────────────────────────────────────────────────
print("=" * 55)
print("Translating with GPT-5.2-chat ...")
print("=" * 55)
gpt_df = run_translation("gpt", OUT_GPT)
gpt_df.head(6)

Translating with GPT-5.2-chat ...
  [gpt] 20/600 ...
  [gpt] 40/600 ...
  [gpt] 60/600 ...
  [gpt] 80/600 ...
  [gpt] 100/600 ...
  [gpt] 120/600 ...
  [gpt] 140/600 ...
  [gpt] 180/600 ...
  [gpt] 200/600 ...
  [gpt] 240/600 ...
  [gpt] 260/600 ...
  [gpt] 300/600 ...
  [gpt] 320/600 ...
  [gpt] 340/600 ...
  [gpt] 360/600 ...
  [gpt] 380/600 ...
  [gpt] 400/600 ...
  [gpt] 420/600 ...
  [gpt] 440/600 ...
  [gpt] 460/600 ...
  [gpt] 480/600 ...
  [gpt] 500/600 ...
  [gpt] 520/600 ...
  [gpt] 540/600 ...
  [gpt] 560/600 ...
  [gpt] 600/600 ...
[gpt] Done — 533 total rows saved → c:\Users\Nguyen Ngo\Downloads\Thesis Quynh\outputs\translations_gpt.csv


,id,source_lang,source_text,translated_text,model
0,1,Spanish,El telefrico ofrece acceso rpido a vistas es...,The cable car offers quick access to spectacul...,gpt
1,1,French,Le tlphrique offre un accs rapide  des vu...,The cable car offers quick access to spectacul...,gpt
2,1,Italian,La funivia offre un rapido accesso a viste spe...,The cable car offers quick access to spectacul...,gpt
3,2,Spanish,Un museo en la cima de un antiguo fuerte y tam...,A museum at the top of an old fort and also a ...,gpt
4,2,French,Un muse au sommet d'un vieux fort et un bon r...,A museum at the top of an old fort and a good ...,gpt
5,2,Italian,Un museo in cima in un vecchio forte e anche u...,A museum on top in an old fort and also a good...,gpt


In [7]:
# ── Run Gemini ─────────────────────────────────────────────────────────────────
print("=" * 55)
print("Translating with Gemini-2.5-flash ...")
print("=" * 55)
gemini_df = run_translation("gemini", OUT_GEMINI)
gemini_df.head(6)

Translating with Gemini-2.5-flash ...
  [gemini] 20/600 ...
  [gemini] 40/600 ...
  [gemini] 60/600 ...
  [gemini] 80/600 ...
  [gemini] 100/600 ...
  [gemini] 120/600 ...
  [gemini] 140/600 ...
  [gemini] 180/600 ...
  [gemini] 200/600 ...
  [gemini] 240/600 ...
  [gemini] 260/600 ...
  [gemini] 300/600 ...
  [gemini] 320/600 ...
  [gemini] 340/600 ...
  [gemini] 360/600 ...
  [gemini] 380/600 ...
  [gemini] 400/600 ...
  [gemini] 420/600 ...
  [gemini] 440/600 ...
  [gemini] 460/600 ...
  [gemini] 480/600 ...
  [gemini] 500/600 ...
  [gemini] 520/600 ...
  [gemini] 540/600 ...
  [gemini] 560/600 ...
  [gemini] 600/600 ...
[gemini] Done — 533 total rows saved → c:\Users\Nguyen Ngo\Downloads\Thesis Quynh\outputs\translations_gemini.csv


,id,source_lang,source_text,translated_text,model
0,1,Spanish,El telefrico ofrece acceso rpido a vistas es...,The cable car offers quick access to spectacul...,gemini
1,1,French,Le tlphrique offre un accs rapide  des vu...,The cable car offers quick access to spectacul...,gemini
2,1,Italian,La funivia offre un rapido accesso a viste spe...,The cable car offers quick access to spectacul...,gemini
3,2,Spanish,Un museo en la cima de un antiguo fuerte y tam...,A museum on top of an old fort and also a good...,gemini
4,2,French,Un muse au sommet d'un vieux fort et un bon r...,A museum at the top of an old fort and a good ...,gemini
5,2,Italian,Un museo in cima in un vecchio forte e anche u...,A museum at the top of an old fort and also a ...,gemini


In [8]:
# ── Quick sanity check ─────────────────────────────────────────────────────────
for label, df in [("GPT", gpt_df), ("Gemini", gemini_df)]:
    print(f"\n── {label} ──")
    print(df["source_lang"].value_counts())
    errors = df[df["translated_text"].str.startswith("ERROR", na=False)]
    print(f"Translation errors: {len(errors)}")
    display(df.sample(min(3, len(df)), random_state=42))


── GPT ──
source_lang
French     200
Italian    200
Spanish    133
Name: count, dtype: int64
Translation errors: 2


,id,source_lang,source_text,translated_text,model
6,3,Spanish,"El casco antiguo de Dubrovnik es mgico, y en ...","Dubrovnik’s Old Town is magical, and nowhere i...",gpt
489,183,Italian,Essendo ancora in crociera a 30 minuti dall'ar...,Still cruising 30 minutes before arriving at t...,gpt
104,35,Italian,"Sebbene non abbia la testa per le altezze, mi ...","Although I don’t have a head for heights, I re...",gpt



── Gemini ──
source_lang
French     200
Italian    200
Spanish    133
Name: count, dtype: int64
Translation errors: 0


,id,source_lang,source_text,translated_text,model
6,3,Spanish,"El casco antiguo de Dubrovnik es mgico, y en ...","Dubrovnik's Old Town is magical, and nowhere i...",gemini
489,183,Italian,Essendo ancora in crociera a 30 minuti dall'ar...,"Still cruising 30 minutes from port, we decide...",gemini
104,35,Italian,"Sebbene non abbia la testa per le altezze, mi ...","Although I'm not good with heights, I really e...",gemini
